In [1]:
import sys
import os

base_folder = "/Users/wangleijie/Documents/PhD/Research/filterbuddy/experiments-web/"
sys.path.append(os.path.join(base_folder, 'experimentweb'))
os.environ['DJANGO_SETTINGS_MODULE'] = 'experimentweb.settings'
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import django
django.setup()

/Users/wangleijie/Documents/PhD/Research/filterbuddy/experiments-web/experiment/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO 2024-06-13 18:34:36,430 core Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
INFO 2024-06-13 18:34:36,617 common Downloaded file to /Users/wangleijie/stanza_resources/resources.json
INFO 2024-06-13 18:34:37,104 core Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

INFO 2024-06-13 18:34:37,104 core Using device: cpu
INFO 2024-06-13 18:

In [2]:
from sharedsteps.models import Participant
from systems.llm_filter import LLMFilter
from sharedsteps.utils import calculate_algorithm_metrics
import numpy as np
import pandas as pd
from scipy import stats

# Function to calculate confidence interval
def confidence_interval(data, confidence=0.95):
    mean = np.mean(data)
    sem = stats.sem(data)  # Standard error of the mean
    margin_of_error = sem * stats.t.ppf((1 + confidence) / 2., len(data) - 1)
    return mean, mean - margin_of_error, mean + margin_of_error


def compare_string_list(list1, list2):
    """
        check if two lists are the same
    """
    if len(list1) != len(list2):
        return False
    for i in range(len(list1)):
        if list1[i] != list2[i]:
            return False
    return True

def find_prompts(prompt, prior_prompts):
    """
        find whether there are exactly the same prompts in the prior prompts
    """
    same_prompts = []
    for prior_prompt in prior_prompts:
        if prior_prompt["rubric"] == prompt["rubric"] and compare_string_list(prior_prompt["positives"], prompt["positives"]) and compare_string_list(prior_prompt["negatives"], prompt["negatives"]):
            same_prompts.append(prior_prompt)
    return same_prompts[0] if len(same_prompts) > 0 else None
        
def get_system(participant_id, system_name="examplesML"):
    participant = Participant.objects.get(participant_id=participant_id)
    condition = participant.get_condition(system_name)
    if condition:
        systems = condition.get_all_systems("build")
        return systems
    return []

In [3]:
def print_promt(prompt):
    
    positives = '\n\t'.join(prompt['positives']) if len(prompt['positives']) > 0 else ""
    negatives = '\n\t'.join(prompt['negatives']) if len(prompt['negatives']) > 0 else ""
    print(f"ID: {prompt['id']}\tName: {prompt['name']}\t {prompt['rubric']}\n\tPositives: {positives}\n\tNegatives: {negatives}")
    
def print_systems(system_name, participant_id, stage="build"):
    participant = Participant.objects.get(participant_id=participant_id)
    condition = participant.get_condition(system_name)
    systems = condition.get_all_systems(stage)
    print(f"there are {len(systems)} systems at {stage} for {participant_id}")
    if system_name == "promptsLLM":
        for system in systems:
            print(f"system at {system.spent_time / 1000}")
            prompts = system.read_prompts()
            print(f"There are {len(prompts)} prompts at {system.spent_time / 1000}")
            for prompt in prompts:
                print_promt(prompt)

In [4]:
# read the third column from the csv "experiment.csv"
eligible_participants = []
ineligible_participants = ["TWL7IJIP0Y", "1AML9EQ1DZ"]
with open(os.path.join(base_folder, "surveys/experiment.csv"), "r") as f:
    lines = f.readlines()
    print(f"participant\texamples\trules\tprompts")
    for line in lines[1:]:
        items = line.split(",")
        participant_id = items[2].strip()
        if len(participant_id.strip()) > 0:
            eligible_participants.append(items[2])
            # examine its logged system
            example_systems = get_system(participant_id, "examplesML")
            prompt_systems = get_system(participant_id, "promptsLLM")
            rules_systems = get_system(participant_id, "rulesTrees")
            print(f"{participant_id}\t{len(example_systems)}\t{len(rules_systems)}\t{len(prompt_systems)}")
eligible_participants = [participant for participant in eligible_participants if participant not in ineligible_participants]
print(f"there are {len(eligible_participants)} eligible participants")

participant	examples	rules	prompts
VEYNZ6NCSA	30	20	31
Z0L0OA6OGP	21	22	23
V8XNOW79LU	31	31	31
1JCOQ98D9F	31	31	24
TWL7IJIP0Y	28	27	19
5FT58ZLTN5	26	30	26
1AML9EQ1DZ	31	31	31
89F8JUYF0M	28	28	25
2I7084ZX0A	31	31	28
PLU5GMNIXZ	31	31	29
HF29MYNNSM	28	30	27
3VE6KIFIXW	31	31	31
V95DL3LLZY	31	31	31
A7JOWMHUO8	30	25	30
0ARIFPWGGA	26	25	25
ZZHLMCQI4C	28	25	25
SKYNRQGQHK	28	29	25
82UU3MFWAJ	29	25	25
MH4KW8L7F8	30	26	27
IT6MNGQFE1	26	25	26
F5S6U5ATJY	30	25	28
0E85ZYLUYG	28	25	25
UY00IB8KK9	30	26	28
ISQK98KK08	25	25	25
XX7MHWDGF4	29	25	28
PL2LV4A1V6	29	26	26
FRS796GHNA	30	25	26
CZNX6L4XDD	30	27	25
B66K6IIXEX	26	25	25
K8QQI40HND	30	25	25
6VM7X2QXEM	31	31	27
EN7NH5ZZEU	26	25	25
RLNQH6KQGZ	27	31	27
DPU61VG4FM	26	27	27
AUASU16PAI	25	31	36
5JJM4HHFVC	25	25	25
PHVZZHDI5I	25	25	25
there are 35 eligible participants


In [5]:
def generate_prompt(dataset, prompt):
    system_prompt = f"""
            For each text in the dataset, give a 1 (True) or 0 (False) prediction to represent whether the text satisfies the description in the rubrics. 
            Each text starts with “DATA” and a number. Both the number and the text are enclosed by “<” and “>”. 
            
            In the following, the user will provide one rubric to help you make your decision. 
            If the given rubric is completely satisfied, give a True prediction. Otherwise, give a False prediction. 
            RETURN YOUR ANSWER in the json format {{“results”: [(index, prediction), ...]}} where (index, prediction) is a tuple, index is the number of the text in the dataset, and prediction is either 1 or 0.
        """

    rubric = f"Rubric: <{prompt['rubric']}>\n"

    # we assume there is only one positive and one negative example for each prompt
    if len(prompt["positives"]) > 0:
        positive_examples = f"\tExamples that should be marked by this rubric as True:"
        for index, example in enumerate(prompt["positives"]):
            positive_examples += f" {index}. <{example}>; "
        rubric += f"{positive_examples}\n"

    if len(prompt["negatives"]) > 0:
        negative_examples = f"\tExamples that should be marked by this rubric as False:"
        for index, example in enumerate(prompt["negatives"]):
            negative_examples += f" {index}. <{example}>; "
        rubric += f"{negative_examples}\n"
    
    user_prompt = f"""\t### RUBRIC\n\t{rubric}"""

    dataset_list = []
    for index in range(len(dataset)):
        text = dataset[index] # we have already cleaned the text in the prepare.py by removing the double quotes
        dataset_list.append(f'DATA<{index}>: <{text}>')    
    batch_str = "\n".join(dataset_list)
    user_prompt = user_prompt + f"""\n\n\t### DATASETS:\n\t{batch_str}"""
    return user_prompt, system_prompt



def prompt_classify(participant_id):
    system_name = "promptsLLM"
    stage = "build"

    participant = Participant.objects.get(participant_id=participant_id)
    condition = participant.get_condition(system_name)
    final_system = condition.get_latest_system(stage)
    groundtruths = condition.get_groundtruth_dataset(stage)

    X_test = [datum["text"] for datum in groundtruths]
    y_test = [datum["label"] for datum in groundtruths]
    
    prompts = final_system.read_prompts()
  

    llm_filter = LLMFilter(prompts, debug=False, retry=True)
    results = llm_filter.test_model(X=X_test, y=y_test)
    texts_predictions = results["texts_predictions"]
        
    prediction = [None for _ in range(len(groundtruths))] # overall predictions
    for index in range(len(groundtruths)):
        text_pred = texts_predictions[index]
        # if at least one text_pred["prediction"] is 1 then the overall prediction is 1
        prediction[index] = 1 if any([pred["prediction"] == 1 for pred in text_pred]) else 0
        for prompt_index in range(len(text_pred)):
            groundtruths[index]["prompt_" + str(prompt_index)] = text_pred[prompt_index]["prediction"]
        groundtruths[index]["prediction"] = prediction[index]
    performance = calculate_algorithm_metrics(y_test, prediction)
    return performance, prediction, groundtruths, prompts

### Prompt for multiple times
Here we experimente with prompting the LLM for five times and observe whether this leads to a better performance

In [6]:
repeat_num = 5
def repeated_prompt(participant_id): 
    repeated_predictions = []
    repeated_performances = []
    for _ in range(repeat_num):
        performance, prediction, groundtruths, prompts = prompt_classify(participant_id)
        repeated_predictions.append(prediction)
        repeated_performances.append(performance)

    information = {}
    # calculate the average uncertainty of predictions
    uncertainty_list = []
    for datum_index in range(len(groundtruths)):
        predictions = [repeated_predictions[i][datum_index] for i in range(repeat_num)]
        mean_predictions = np.mean(predictions)
        uncertainty = mean_predictions * (1 - mean_predictions)
        uncertainty_list.append(uncertainty)
    mean, ci_lower, ci_upper = confidence_interval(uncertainty_list)
    print(f"uncertainty mean: {mean:.3f}, CI: ({ci_lower:.3f}, {ci_upper:.3f})")
    information["uncertainty"] = {"mean": mean, "ci_lower": ci_lower, "ci_upper": ci_upper}


    # calculate variance of the performances
    for metric in ["accuracy", "precision", "recall", "f1"]:
        values = [performance[metric] for performance in repeated_performances]
        mean, ci_lower, ci_upper = confidence_interval(values)
        print(f"{metric} mean: {mean:.3f}, CI: ({ci_lower:.3f}, {ci_upper:.3f})")
        information[metric] = {"mean": mean, "ci_lower": ci_lower, "ci_upper": ci_upper}
    
    # calculate the majority vote
    majority_vote = []
    for datum_index in range(len(groundtruths)):
        predictions = [repeated_predictions[i][datum_index] for i in range(repeat_num)]
        majority_vote.append(1 if np.mean(predictions) > 0.5 else 0)
    y_test = [datum["label"] for datum in groundtruths]
    majority_vote_performance = calculate_algorithm_metrics(y_test, majority_vote)
   
    repeated_performances.append(majority_vote_performance)
    header = "Index\tAccuracy\tPrecision\tRecall\tF1"
    print(header)
    for index, performance in enumerate(repeated_performances):
        print(f"{index}\t{performance['accuracy']:.3f}\t{performance['precision']:.3f}\t{performance['recall']:.3f}\t{performance['f1']:.3f}")
    
    information["majority_vote"] = majority_vote_performance
    return information

In [ ]:
participant_information = {}
for participant_id in eligible_participants[:20]:
    information = repeated_prompt("promptsLLM", participant_id)
    participant_information[participant_id] = information

In [ ]:
# convert to a pandas dataframe
information = []
for participant_id, info in participant_information.items():
    for metric in info:
        if metric == "majority_vote":
            continue
        
        information.append({
            "participant_id": participant_id, 
            "metric": metric, 
            "mean": info[metric]["mean"], 
            "ci_lower": info[metric]["ci_lower"], 
            "ci_upper": info[metric]["ci_upper"],
            "majority_vote": info["majority_vote"][metric] if metric != "uncertainty" else 0
        })
information_df = pd.DataFrame(information)
information_df.drop(columns=["participant_id"]).groupby("metric").mean()


In [7]:
participant_id = "V95DL3LLZY" # "5FT58ZLTN5"
performance, prediction, groundtruths, prompts = prompt_classify(participant_id)

In [112]:
for prompt in prompts:
    print_promt(prompt)

ID: 0	Name: explicit language	 Remove texts that contain explicit language (e.g., fuck, bitchesor comparable slurs) and texts that demean a persons intelligence. Watch for words like idiot, stupid, etc.
	Positives: Only in America, your country is fuckt up,.
	
	
	Negatives: No other Country in the world has a Constitution that sets out rights. And the 2nd amendment id about a personal right to own a firearm. If you do not like that 2nd Amendment or the Electoral collage then gather your people and propose a Constitutional Amendment.
ID: 1	Name: dehumanizing people	 Remove texts that dehumanize individuals by comparing them to excrement or non-human entities.
	Positives: This OLD fart is not HUMAN PURE GARBAGE
	Negatives: 
ID: 2	Name: Self harm	 Remove texts that explicitly encourage or describe acts of self-harm or suicide.
	Positives: 
	Negatives: How pro-life are u if your willing to shoot your own grandchildren
	kill a Latino 9 yr old its ok. disgusting
	


In [113]:
for datum in groundtruths:
    if datum["prediction"] != datum["label"]:
        print("=" * 20)
        print(f"Text: {datum['text']}")
        print(f"Label: {datum['label']}\t" + "\t".join([f"Prompt_{index}: {datum['prompt_' + str(index)]}" for index in range(len(prompts))]))

Text: Yet another politician whose stupidity is only surpassed by those who vote for her and will belive ANYTHING that sounds pro gun. Meanwhile, I wonder if her son or daughter could be reached for comment?
Label: 0	Prompt_0: 1	Prompt_1: 0	Prompt_2: 0
Text: Why are these republicunts so gross 🤮.. gun owners are ugly inside outside
Label: 0	Prompt_0: 1	Prompt_1: 0	Prompt_2: 0
Text: this is what happens when you allow stupid to run wild with no restrictions. that guy should be tried for murder and be sentenced to life
Label: 0	Prompt_0: 1	Prompt_1: 0	Prompt_2: 0
Text: Bad take.  For you to claim that dark money controls politics and than mock right wingers for claiming secret cabals control politics, is just intellectually dishonest Texas horseshite.
Label: 1	Prompt_0: 0	Prompt_1: 0	Prompt_2: 0
Text: Chicago, and ny are shit holes with massive gun control due to criminals getting illegal firearms.
Label: 0	Prompt_0: 0	Prompt_1: 1	Prompt_2: 0
Text: Why is the taliban party so obsessed wi

In [9]:
from sharedsteps import utils
import copy, re

def wrap_in_xml(datapoint, ignore, level=0):
    # flatten a dict in the format of xml strings in recursive way
    xml_str = ""
    indent = "    " * level  # 4 spaces per indentation level

    for key, value in datapoint.items():
        if ignore and key in ignore:
            continue
        if isinstance(value, dict):
            xml_str += f"{indent}<{key}>\n"
            xml_str += wrap_in_xml(value, ignore, level + 1)
            xml_str += f"{indent}</{key}>\n"
        else:
            xml_str += f"{indent}<{key}>{value}</{key}>\n"
    
    return xml_str

def extract_xml(xml_str, tag):
    # extract the value of the tag from the xml string using regular expression
    pattern = re.compile(fr"<{tag}>(.*?)</{tag}>", re.DOTALL)
    result = pattern.findall(xml_str)
    return result[0].strip() if len(result) > 0 else None


def prompt_wrapper(system_prompt, datapoint, ignore=None, printed=True):
    chatbot = utils.ChatCompletion()
    user_prompt = wrap_in_xml(datapoint, ignore)
    if printed:
        print(user_prompt)
    response = chatbot.chat_completion(
        system_prompt = system_prompt,
        user_prompt = user_prompt,
        type="text"
    )
    return response


def select_comment(test_data, i, j):
    # select the j-th comment from the i-th prompt
    now_data = test_data[i]
    datapoint = {}
    datapoint["Prompt"] = copy.deepcopy(now_data["Prompt"])
    jth_comment = now_data["Comments"][j]
    datapoint.update(copy.deepcopy(jth_comment))
    return datapoint

In [10]:
test_data = [
    {
        "Prompt": {
            "Rubric": "Remove comments that use derogatory terms such as stupid, idiot, losers to insult individuals.",
            "RemovedExample": "She is a total idiot but in fairness, by shooting them she meant the perpetrators, but like Trump, she cannot admit to even a simple misstatement."
        },
        "Comments": [
            {"Groundtruth": 0, "Text": "That is one stupid grand jury. I guess a civil suit is the only avenue left open for justice for that poor little girl. The only legal one that is"},
            {"Groundtruth": 1, "Text": "Please ensure American idiots like this don't have passports. People in the real world don't want to be infected by their stupidity."}
        ]
    },
    {
        "Prompt": {
            "Rubric": "Remove texts that stereotypes political parties and puts them in bad lights.",
        },
        "Comments": [
            {"Groundtruth": 0, "Text": "Republican control states are more dangerous than Democrats control states."},
            {"Groundtruth": 0, "Text": "I remember when conservatives thought the purge would be in Illinois because suspects in non-violent crimes don't have to pay bail and get to continue living life while waiting for their court date. Meanwhile they do shit like this and they're cool with it."},
            {"Groundtruth": 0,  "Text": "The left needs to strap up because if the reich wing takes over, death squads will be a thing with those people."},
            {"Groundtruth": 0, "Text": "Once again, the Republican party aren't sending their best. Or worse, they are."}
        ]
    },
    {
        "Prompt": {
            "Rubric": "Remove texts that talks about a person killing another person or a person killing themselves.",
        },
        "Comments": [
            {"Groundtruth": 0, "Text": "If that was my grandmother after that crazy 🤪 comment, I would tell her that you are dead to me, and I would never want to hear from you again."},
            {"Groundtruth": 0, "Text": "Texans shooting Texans. Okay fine, I’ll care as much as I’m obliged to. But like…"},
            {"Groundtruth": 0, "Text": "Wait until the criminal has a gun to your head, and the police are taking a long time to arrive to help you. Well since you don’t have a gun and no one can help you…. Sorry RIP"},
            {"Groundtruth": 0, "Text": "Findings show republicans are not trying to do harm when they shoot themselves or their grandchildren. but a new study has concluded they are actually trying to protect everyone .. 🤪"},
            {"Groundtruth": 0, "Text": "Well, everyone has a Freudian slip now and then. Still, looking forward to having my head blown clean off by a law abiding citizen with an arsenal of artillery who had a bad day."},
            {"Groundtruth": 0, "Text": "“I’ll protect my grandkids but killing them.” Whoever the daughter or son of that crazy lady should not send their kids to her place anymore"}
        ]
    },
]

def classify_comment(datapoint):
    system_prompt = """
        You will play a role of content moderators and decide whether the text should be removed or not according to the given prompt. 
        For the given text, give a 1 (Removed) or 0 (Not removed) prediction.

        ### TASK DESCRIPTION
        Each prompt include a rubric and an optional list of examples that should be removed or not removed for reference.
        You should first reason over whether the texts should be removed or not according to the rubric.
        If you think the text should be removed, give a 1 (Removed) prediction. Otherwise, give a 0 (Not removed) prediction.
        
        ### EXAMPLE
            #### INPUT
            <Prompt>
                <Rubric>
                    Remove comments that use derogatory terms such as stupid, idiot, losers to insult individuals.
                </Rubric>
                <RemovedExample>She is a total idiot </RemovedExample>
            </Prompt>
            <Text>Please ensure American idiots like this don't have passports. People in the real world don't want to be infected by their stupidity.</Text>

            #### OUTPUT
            <Reasoning>
                The text uses the derogatory term "idiots" to insult a specific individual as the phrase "like this" indicates that the speaker is referring a specific individual
                Therefore, the text should be removed.
            </Reasoning>
            <Decision>1</Decision>
    """
    response = ""
    print("=" * 50 + "PREDICTION" + "=" * 50)
    print(f"<Groundtruth>{datapoint['Groundtruth']}</Groundtruth>")
    first_time = True
    while True:
        response = prompt_wrapper(system_prompt, datapoint, ignore=["Groundtruth"], printed=first_time)
        first_time = False
        decision = extract_xml(response, "Decision")
        if decision == "1" or decision == "0":
            print("-" * 60)
            print(response)
            return int(decision)

In [11]:
datapoint = select_comment(test_data, 0, 1)
decision = classify_comment(datapoint)


==================================================PREDICTION==================================================
<Groundtruth>1</Groundtruth>
<Prompt>
    <Rubric>Remove comments that use derogatory terms such as stupid, idiot, losers to insult individuals.</Rubric>
    <RemovedExample>She is a total idiot but in fairness, by shooting them she meant the perpetrators, but like Trump, she cannot admit to even a simple misstatement.</RemovedExample>
</Prompt>
<Text>Please ensure American idiots like this don't have passports. People in the real world don't want to be infected by their stupidity.</Text>

------------------------------------------------------------
<Reasoning>
The text uses the derogatory term "idiots" to insult a specific group of individuals, which is in violation of the rubric's instruction to remove comments with derogatory terms like "stupid," "idiot," or "losers" that are used to insult individuals.
Therefore, the text should be removed.
</Reasoning>
<Decision>1</Decisi

(1, 1)

In [12]:
# Criteria for a good prompt
"""
    - Is a follow-up question always needed? sometimes it seems to me that the reason of misclassification is clear enough
    - Make the follow-up question easy to answer. The follow-up question should be grounded in details, for now it often poses a very high-level question that is hard to answer for users.
    - Only make necessary changes to the prompt. 
    - Sometimes even though the agent has a correct reasoning, the improved prompt leads to the opposite direction. --> should add some checkers to ensure the prompt is improved
"""

def refine_prompt(datapoint):
    # refine the prompt to make it more readable
    system_prompt = """
        A user is writing down their content moderation preferences in prompts but has difficulties in clearly communicating their preferences in prompts. 
        However, they could intuitively tell the groundtruth of a text (1 represents the text should be removed, and 0 represents the text should not be removed).
        The current prompt have incorrectly classified the given example.
        Your task is to help improve the prompt according to the example and the ground truth label.

        ### TASK DESCRIPTION
        You should first reason why the prompt incorrectly classified the given example.
        Then you should ask a simple follow-up yes-no question in the format of "Do you want to remove/keep comments because it ....?" so that you could your reasoning.
        After that, you should suggest how to improve this prompt based on your reasoning. 
        Your should only make changes to the original prompt that are necessary.  You should only make changes under 10 words.
        Your follow-up question is intended for generic social media users and therefore should be simple, concise. 

        ### EXAMPLE 1
        #### INPUT
        <Prompt><Rubric>Remove texts that talk about a person killing another person</Rubric></Prompt>
        <Text>I want to commit suicide</Text>
        <Groundtruth>1</Groundtruth>

        #### OUTPUT
        <Reasoning>
            The positive example suggests that the user also wants to remove suicide events rather than simply a person killing another person. 
            Therefore, the prompt might classify the example incorrectly. 
        </Reasoning>
        <Follow-up>
            Do you want to remove comments because it also mentions a person killing themselves?
        </Follow-up>
        <ImprovedPrompt>
            Remove texts that talk about a person killing another person or a person killing themselves
        </ImprovedPrompt>


        ### EXANPLE 2
        #### INPUT
        <Prompt><Rubric>Remove comments that use derogatory terms such as stupid, idiot, losers to insult individuals.</Rubric></Prompt>
        <Text>That is one stupid grand jury. I guess a civil suit is the only avenue left open for justice for that poor little girl. The only legal one that is</Text>
        <Groundtruth>0</Groundtruth>

        #### OUTPUT
        <Reasoning>
            The negative example suggests that the user might still want to see comments that use derogatory terms against a group or an entity.
            Therefore, the prompt might classify the example incorrectly. 
        </Reasoning>
        <Follow-up>
            Do you still want to keep this comment because it insult a group of people like a jury but not specific individuals?
        </Follow-up>
        <ImprovedPrompt>
            Remove comments that use derogatory terms such as stupid, idiot, losers to directly insult specific individuals.
        </ImprovedPrompt>
    """
    
    response = ""
    print("=" * 50 + "REFINEMENT" + "=" * 50)
    first_time = True
    while True:
        response = prompt_wrapper(system_prompt, datapoint, ignore=["RemovedExample"], printed=first_time)
        first_time = False
        improved_prompt = extract_xml(response, "ImprovedPrompt")
        if improved_prompt is not None:
            print("-" * 60)
            print(response)
            return improved_prompt

In [13]:
datapoint = select_comment(test_data, 0, 0)
decision = classify_comment(datapoint)
while decision != datapoint["Groundtruth"]:
    print("The decision is incorrect")
    improved_prompt = refine_prompt(datapoint)
    datapoint["Prompt"]["Rubric"] = improved_prompt
    decision = classify_comment(datapoint)

==================================================PREDICTION==================================================
<Groundtruth>0</Groundtruth>
<Prompt>
    <Rubric>Remove comments that use derogatory terms such as stupid, idiot, losers to insult individuals.</Rubric>
    <RemovedExample>She is a total idiot but in fairness, by shooting them she meant the perpetrators, but like Trump, she cannot admit to even a simple misstatement.</RemovedExample>
</Prompt>
<Text>That is one stupid grand jury. I guess a civil suit is the only avenue left open for justice for that poor little girl. The only legal one that is</Text>

------------------------------------------------------------
<Reasoning>
The text uses the derogatory term "stupid" to insult a group of people (the grand jury). The rubric specifies that comments using derogatory terms to insult individuals should be removed. Since "grand jury" is a collective term for a group of individuals, it falls within the scope of the rubric's instructi




### DATASETS:
DATA<0>: <That is one stupid grand jury. I guess a civil suit is the only avenue left open for justice for that poor little girl. The only legal one that is.>